# NB_AGT_03 · Evaluar el agente y publicar si aprueba

Corre las preguntas `VALIDADA` con uso `EVALUACION` contra el agente, compara cada respuesta con la que validó el área y decide:

| Condición | Resultado |
|---|---|
| Acierto ≥ `UMBRAL_ACIERTO` **y** no cae más de `MAX_CAIDA_PP` puntos contra la última evaluación aprobada | Aprueba. Si `PUBLICAR_SI_APRUEBA = True` y `ETAPA = 'sandbox'`, publica el borrador |
| Cualquier otra cosa | No aprueba. No publica |

Deja:

- Una fila en `agente_ctl.eval_historial` por corrida (el tablero de evolución sale de acá).
- `Files/agente/<AGENTE>/evaluaciones/<fecha>_<evaluation_id>_fallas.csv`, con las preguntas que fallaron. Ese archivo es la entrada de la skill `agente-datos-fabric` (tarea C) y de la revisión con el área.

`ETAPA = 'sandbox'` evalúa el **borrador** (lo que dejó `NB_AGT_02`). `ETAPA = 'production'` evalúa lo publicado: usalo en la corrida programada para detectar que el agente empeoró sin cambios (datos nuevos, cambios de esquema, cambios del modelo de Fabric).

SDK de evaluación: [preview](https://learn.microsoft.com/en-us/fabric/data-science/evaluate-data-agent). Probado contra el código de `fabric-data-agent-sdk 0.1.33a0`, no contra un agente real.

In [ ]:
%pip install -q fabric-data-agent-sdk==0.1.33a0

In [ ]:
AGENTE = "AG_VENTAS"
WORKSPACE = None
TABLA = "agente_ctl.banco_preguntas"
TABLA_HISTORIAL = "agente_ctl.eval_historial"
TABLA_RESULTADOS = "eval_agente"     # tablas del SDK: <nombre> y <nombre>_steps. Solo letras, números y _
ETAPA = "sandbox"                    # "sandbox" = borrador; "production" = publicado
UMBRAL_ACIERTO = 0.85
MAX_CAIDA_PP = 2.0                   # puntos porcentuales que se toleran contra la última aprobada
PUBLICAR_SI_APRUEBA = False
DESCRIPCION_PUBLICACION = ""         # vacío = la arma el notebook con la fecha y el resultado

In [ ]:
# --- funciones (sin Fabric: las usa pruebas/prueba_local.py) ---
import math

# El SDK decide con una regla simple sobre la respuesta del juez: si contiene "no" -> incorrecta,
# si contiene "yes" -> correcta (ver evaluation/_evaluation_runner.py del SDK). Por eso el juez
# tiene que contestar SOLO "yes" o "no", en inglés, aunque las preguntas estén en español:
# un "sí" no se reconoce y cualquier palabra con "no" adentro ("número") la marca incorrecta.
CRITIC_PROMPT = """You are checking a business data assistant. The question and answers may be in Spanish.

Question: {query}

Expected answer (validated by the business area):
{expected_answer}

Actual answer:
{actual_answer}

Rules:
- Numbers are equivalent if they match after rounding to the precision of the expected answer.
  Ignore thousands separators, currency symbols and decimal comma vs decimal point.
- Lists and rankings must contain the same items in the same order.
- Extra explanation in the actual answer is fine if the data matches.
- If the actual answer asks for clarification or says it cannot answer, it is not equivalent.

Reply with exactly one lowercase word: yes or no."""


def resumen_corrida(detalle) -> dict:
    """detalle: DataFrame de get_evaluation_details(get_all_rows=True).
    evaluation_judgement es True, False o None (el juez no fue claro)."""
    juicio = detalle["evaluation_judgement"]
    total = len(detalle)
    correctas = int((juicio == True).sum())  # noqa: E712 (puede venir como objeto)
    incorrectas = int((juicio == False).sum())  # noqa: E712
    dudosas = total - correctas - incorrectas
    return {"total": total, "correctas": correctas, "incorrectas": incorrectas,
            "dudosas": dudosas, "acierto": round(correctas / total, 4) if total else 0.0}


def decidir(resumen: dict, acierto_anterior, umbral: float, max_caida_pp: float):
    """Devuelve (aprueba, motivos). Las dudosas cuentan como no acertadas."""
    motivos = []
    if resumen["total"] == 0:
        return False, ["No hay preguntas de evaluación validadas"]
    if resumen["acierto"] < umbral:
        motivos.append(f"Acierto {resumen['acierto']:.1%} menor al umbral {umbral:.0%}")
    if acierto_anterior is not None and not (isinstance(acierto_anterior, float) and math.isnan(acierto_anterior)):
        caida = (acierto_anterior - resumen["acierto"]) * 100
        if caida > max_caida_pp:
            motivos.append(f"Cae {caida:.1f} pp contra la última aprobada ({acierto_anterior:.1%})")
    return (not motivos), motivos

In [ ]:
# --- preguntas de evaluación ---
import pandas as pd
evals = spark.sql(f"""
    SELECT pregunta_id, area_negocio, fuente, pregunta, respuesta_esperada
    FROM {TABLA}
    WHERE agente = '{AGENTE}' AND uso = 'EVALUACION' AND estado_efectivo = 'VALIDADA'
""").toPandas()
print(f"{len(evals)} preguntas de evaluación validadas")
df = evals.rename(columns={"pregunta": "question", "respuesta_esperada": "expected_answer"})[["question", "expected_answer"]]

In [ ]:
# --- evaluar (consume capacidad: una conversación por pregunta, más el juez) ---
from fabric.dataagent.evaluation import evaluate_data_agent, get_evaluation_details

evaluation_id = evaluate_data_agent(
    df, AGENTE,
    workspace_name=WORKSPACE,
    table_name=TABLA_RESULTADOS,
    critic_prompt=CRITIC_PROMPT,
    data_agent_stage=ETAPA,
)
detalle = get_evaluation_details(evaluation_id, TABLA_RESULTADOS, get_all_rows=True, verbose=False)
resumen = resumen_corrida(detalle)
print(evaluation_id, resumen)

In [ ]:
# --- decidir contra la última evaluación aprobada ---
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TABLA_HISTORIAL.split('.')[0]}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLA_HISTORIAL} (
        agente STRING, evaluation_id STRING, etapa STRING, evaluado_en TIMESTAMP,
        total INT, correctas INT, incorrectas INT, dudosas INT, acierto DOUBLE,
        aprueba BOOLEAN, motivos STRING, publicado BOOLEAN, banco_actualizado_en TIMESTAMP
    ) USING DELTA""")
anterior = spark.sql(f"""
    SELECT acierto FROM {TABLA_HISTORIAL}
    WHERE agente = '{AGENTE}' AND aprueba ORDER BY evaluado_en DESC LIMIT 1""").collect()
acierto_anterior = anterior[0]["acierto"] if anterior else None
aprueba, motivos = decidir(resumen, acierto_anterior, UMBRAL_ACIERTO, MAX_CAIDA_PP)
print("APRUEBA" if aprueba else "NO APRUEBA", motivos)

In [ ]:
# --- fallas para revisar (con el área y con Claude) ---
import os, datetime as dt
mapa = evals.set_index("pregunta")
fallas = detalle[detalle["evaluation_judgement"] != True].copy()  # noqa: E712
fallas["pregunta_id"] = fallas["question"].map(mapa["pregunta_id"])
fallas["area_negocio"] = fallas["question"].map(mapa["area_negocio"])
fallas["fuente"] = fallas["question"].map(mapa["fuente"])
fallas["resultado"] = fallas["evaluation_judgement"].map({False: "INCORRECTA"}).fillna("DUDOSA")
columnas = ["pregunta_id", "area_negocio", "fuente", "resultado", "question", "expected_answer",
            "actual_answer", "evaluation_message", "thread_url"]
fallas = fallas[[c for c in columnas if c in fallas.columns]]
sello = dt.datetime.now().strftime("%Y%m%d_%H%M")
carpeta = f"/lakehouse/default/Files/agente/{AGENTE}/evaluaciones"
os.makedirs(carpeta, exist_ok=True)
ruta_fallas = f"{carpeta}/{sello}_{evaluation_id}_fallas.csv"
fallas.to_csv(ruta_fallas, index=False, encoding="utf-8-sig")
print(f"{len(fallas)} fallas en {ruta_fallas}")
display(fallas)

In [ ]:
# --- publicar (cambio con impacto: lo ven todos los usuarios del agente) y registrar ---
from fabric.dataagent.client import FabricDataAgentManagement
publicado = False
if aprueba and PUBLICAR_SI_APRUEBA and ETAPA == "sandbox":
    desc = DESCRIPCION_PUBLICACION or f"Banco validado, evaluación {evaluation_id}: acierto {resumen['acierto']:.1%}"
    FabricDataAgentManagement(AGENTE, WORKSPACE).publish_staging(description=desc)
    publicado = True
    print("Publicado.")
elif aprueba:
    print("Aprueba. Para publicar: PUBLICAR_SI_APRUEBA = True con ETAPA = 'sandbox', o publicá desde el portal.")
else:
    print("No se publica. Revisá las fallas antes de cambiar nada.")

banco_ts = spark.sql(f"SELECT max(cargado_en) AS ts FROM {TABLA} WHERE agente = '{AGENTE}'").collect()[0]["ts"]
fila = [(AGENTE, evaluation_id, ETAPA, dt.datetime.now(), resumen["total"], resumen["correctas"],
         resumen["incorrectas"], resumen["dudosas"], float(resumen["acierto"]), aprueba,
         " | ".join(motivos), publicado, banco_ts)]
spark.createDataFrame(fila, spark.table(TABLA_HISTORIAL).schema).write.mode("append").saveAsTable(TABLA_HISTORIAL)

# Para que la corrida programada falle (y avise) si el agente publicado empeoró:
if ETAPA == "production" and not aprueba:
    raise RuntimeError("El agente publicado no aprueba la evaluación: " + " | ".join(motivos))